# C15_E03 — Sandbox: ensayo de retuneo sobre gemelo

**Capítulo:** 15 — Gemelos digitales, modelos híbridos y simulación conectada  
**Identificador:** `C15_E03`  
**Objetivo pedagógico:** Validar un cambio antes de aplicarlo en planta usando un gemelo digital calibrado.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Gemelo de horno usado para ensayar nueva sintonía PI antes de cargarla en DCS.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Sandbox: ensayo de cambio de sintonía sobre gemelo

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct
import os

# Gemelo: FOPDT identificado del horno
K, tau, theta = 0.8, 60.0, 6.0
def pade1(theta): return ct.tf([-theta/2.0, 1.0], [theta/2.0, 1.0])
G = ct.tf([K], [tau, 1.0])*pade1(theta)

# Sintonía actual en planta (Lambda con λ = 2θ)
Kc_old = tau/(K*(2*theta + theta)); Ti_old = tau
PI_old = ct.tf([Kc_old*Ti_old, Kc_old], [Ti_old, 0.0])

# Nueva sintonía propuesta (SIMC con λ = θ)
Kc_new = (1.0/K)*tau/(theta + theta); Ti_new = min(tau, 4*(theta + theta))
PI_new = ct.tf([Kc_new*Ti_new, Kc_new], [Ti_new, 0.0])
print(f"Antigua: Kc={Kc_old:.3f}, Ti={Ti_old:.1f}")
print(f"Nueva  : Kc={Kc_new:.3f}, Ti={Ti_new:.1f}")

Antigua: Kc=4.167, Ti=60.0
Nueva  : Kc=6.250, Ti=48.0


## 2. Comparativa (sandbox antes de aplicar en planta)

In [2]:
t = np.linspace(0, 600, 1500)
T_old = ct.feedback(PI_old*G, 1)
T_new = ct.feedback(PI_new*G, 1)
_, y_old = ct.step_response(T_old, t)
_, y_new = ct.step_response(T_new, t)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_old, label="Sintonía actual (Lambda 2θ)")
ax.plot(t, y_new, '--', label="Sintonía propuesta (SIMC)")
ax.axhline(1.0, ls=':', color='gray')
ax.set_xlabel("t (s)"); ax.set_ylabel("y/SP")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C15_E03 — Sandbox: ensayo de retuneo sobre gemelo")
out = '../../figures/cap15/fig_C15_F04_sandbox.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

El gemelo se usa como sandbox para evaluar el retuneo antes de aplicarlo en planta. Métricas de comparación: tiempo de establecimiento, sobredisparo, IAE en seguimiento y rechazo. La decisión de aplicar se toma cuando la nueva sintonía supera a la actual en métricas relevantes y mantiene márgenes de robustez. **Crítico:** el gemelo debe estar calibrado (CUSUM dentro de umbral) para que el ensayo sea fiable.